<a href="https://colab.research.google.com/github/JoshPardosi-231401031/Python-Data-Cleaning-Project/blob/main/Project_DataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ydata-profiling pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from google.colab import files

In [ ]:
uploaded = files.upload()

In [ ]:
df = pd.read_csv('messy_customer_orders.csv')

In [ ]:
profile = ProfileReport(df, title='Profiling Report', explorative=True)
profile.to_file('report.html')

In [ ]:
# Jumlah Baris dan Kolom
print('data size:', df.shape )

In [ ]:
df.dtypes

In [ ]:
print(f'\nData Kosong:\n{df.isnull().sum()}')

In [ ]:
df.head()

In [ ]:
# Standarize Columns names in one line
df.columns = df.columns.str.replace(' ', '_').str.lower().str.strip()
print(df.columns)

In [ ]:
# Check percentage missing value
missing = df.isnull().mean()*100
print(missing[missing > 0].sort_values(ascending = True))

In [ ]:
# how to handle missing value systematically

# drop duplicates rows
df = df.drop_duplicates()
# fill missing numbers with median
df['total_amount'] = df['total_amount'].fillna(df['total_amount'].median())
# fill mising text with 'unknown'
categorical_cols = ['customer_id', 'country', 'category']
for col in categorical_cols:
  df[col] = df[col].fillna('unknown')
# Check The remaining Missing Values
print(df.isnull().sum())


In [ ]:
# standarized text with mapping dictionary

# 1 change text to lowercase and no extraspace
cols_to_clean =['category', 'status', 'country']
for col in cols_to_clean:
  df[col] = df[col].astype(str).str.strip().str.lower()

# 2 fix the wrong text
country_map = {
    'u.s.a': 'usa',
    'united states' : 'usa',
    'us' : 'usa',
    'd' : 'germany',
    'de' : 'germany',
    'united kingdom' : 'uk',
    'u.k.' : 'uk',
    'aus' : 'australia'
}
# apply dict to country cols
df['country'] = df['country'].map(country_map).fillna(df['country'])
# print
print(df['country'].value_counts())



In [ ]:
# fix and extract the order_date
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# extract useful time features into new columns
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day_of_week'] = df['order_date'].dt.day_name()

print(df[['order_date', 'order_year', 'order_month', 'order_day_of_week']].head())

In [ ]:
# make filter for impossible order quantity
impossible_order_quantity = df['quantity'] <= 0
# check the missing rows
print(df.loc[impossible_order_quantity, ['order_id', 'product', 'quantity', 'unit_price', 'total_amount']])

In [ ]:
# make a new clean csv
df.to_csv('cleaned_customer_orders.csv', index=False)
# import files
from google.colab import files
files.download('cleaned_customer_orders.csv')